# Varroa Mite Detection — Training

Clean, repository-ready training notebook for the Honey Chain Varroa mite detector.

## Workflow
1. Prepare the annotated images in YOLO format.
2. Create the YOLO dataset configuration.
3. Train the final detector.
4. Validate the trained model.
5. Run sample inference.
6. Export/use `best.pt` in the Honey Chain ML service.

This notebook intentionally excludes exploratory experiments, repeated model variants, intermediate outputs, and personal machine paths.


## 1. Imports

In [ ]:
from pathlib import Path
import os
import shutil

from ultralytics import YOLO


## 2. Prepare the YOLO dataset

In [ ]:
from pathlib import Path
import shutil
import csv

# ==========================================================
# FINAL CLEAN YOLO DATASET CREATION
# ==========================================================

DATASET_DIR = Path("/kaggle/input/<dataset-owner>/varroa/varroa")
YOLO_DIR = Path("/kaggle/working/varroa_yolo_final")

# Remove previous dataset if it exists
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

# Create folders
for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

report_dir = YOLO_DIR / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Read gt_one.csv
# Last occurrence wins for duplicate image paths
# ----------------------------------------------------------

annotations = {}

duplicate_rows = []

for split in ["train", "val", "test"]:

    gt_file = DATASET_DIR / split / "gt_one.csv"

    split_dict = {}

    with open(gt_file, "r", encoding="utf-8") as f:

        for line_no, line in enumerate(f, start=1):

            parts = line.strip().split()

            if not parts:
                continue

            img_path = parts[0]
            cls = int(parts[1])
            coords = list(map(float, parts[2:]))

            if img_path in split_dict:
                duplicate_rows.append({
                    "split": split,
                    "path": img_path,
                    "line": line_no
                })

            split_dict[img_path] = {
                "class": cls,
                "coords": coords,
                "line": line_no
            }

    annotations[split] = split_dict

# ----------------------------------------------------------
# Conversion
# ----------------------------------------------------------

stats = {}
skipped_images = []

for split in ["train", "val", "test"]:

    image_root = DATASET_DIR / split / "videos"

    stats[split] = {
        "images": 0,
        "healthy": 0,
        "infected": 0,
        "boxes": 0,
        "skipped": 0
    }

    for image_path in image_root.rglob("*.png"):

        stats[split]["images"] += 1

        relative = image_path.relative_to(DATASET_DIR / split)
        relative_str = relative.as_posix()

        ann = annotations[split][relative_str]

        cls = ann["class"]
        coords = ann["coords"]

        # output path preserving subfolders
        out_img = YOLO_DIR / "images" / split / relative
        out_lbl = YOLO_DIR / "labels" / split / relative.with_suffix(".txt")

        out_img.parent.mkdir(parents=True, exist_ok=True)
        out_lbl.parent.mkdir(parents=True, exist_ok=True)

        # ---------------- Healthy ----------------

        if cls == 0:

            shutil.copy2(image_path, out_img)
            out_lbl.write_text("")

            stats[split]["healthy"] += 1
            continue

        # ---------------- Infected ----------------

        if cls not in [1, 3]:
            skipped_images.append({
                "split": split,
                "image": relative_str,
                "reason": "Unknown class"
            })
            stats[split]["skipped"] += 1
            continue

        # coordinates must be groups of 4
        if len(coords) == 0 or len(coords) % 4 != 0:
            skipped_images.append({
                "split": split,
                "image": relative_str,
                "reason": "Malformed coordinates"
            })
            stats[split]["skipped"] += 1
            continue

        lines = []
        valid = True

        for i in range(0, len(coords), 4):

            x1, y1, x2, y2 = coords[i:i+4]

            if not (0 <= x1 < x2 <= 160 and
                    0 <= y1 < y2 <= 280):

                valid = False
                break

            xc = ((x1+x2)/2)/160
            yc = ((y1+y2)/2)/280
            w  = (x2-x1)/160
            h  = (y2-y1)/280

            lines.append(
                f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
            )

        # skip entire corrupted image
        if not valid:

            skipped_images.append({
                "split": split,
                "image": relative_str,
                "reason": "Invalid bounding box"
            })

            stats[split]["skipped"] += 1
            continue

        shutil.copy2(image_path, out_img)
        out_lbl.write_text("\n".join(lines))

        stats[split]["infected"] += 1
        stats[split]["boxes"] += len(lines)

# ----------------------------------------------------------
# Reports
# ----------------------------------------------------------

with open(report_dir/"skipped_images.csv","w",newline="") as f:
    writer=csv.DictWriter(
        f,
        fieldnames=["split","image","reason"]
    )
    writer.writeheader()
    writer.writerows(skipped_images)

with open(report_dir/"duplicate_rows.csv","w",newline="") as f:
    writer=csv.DictWriter(
        f,
        fieldnames=["split","path","line"]
    )
    writer.writeheader()
    writer.writerows(duplicate_rows)

# ----------------------------------------------------------
# data.yaml
# ----------------------------------------------------------

yaml=f"""
path: {YOLO_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: Varroa_mite
"""

(YOLO_DIR/"data.yaml").write_text(yaml.strip())

# ----------------------------------------------------------
# Final summary
# ----------------------------------------------------------

print("="*65)
print("FINAL YOLO DATASET SUMMARY")
print("="*65)

for split in ["train","val","test"]:

    s=stats[split]

    print(f"""
{split.upper()}
Images processed : {s['images']}
Healthy          : {s['healthy']}
Infected         : {s['infected']}
YOLO boxes       : {s['boxes']}
Skipped images   : {s['skipped']}
""")

print("="*65)
print("Duplicate gt_one rows :",len(duplicate_rows))
print("Skipped images        :",len(skipped_images))
print("Dataset location:")
print(YOLO_DIR)
print("="*65)

## 3. YOLO dataset configuration

In [ ]:
from pathlib import Path

DATASET_DIR = Path(
    "/kaggle/input/<dataset-owner>/varroa-yolo-final/varroa_yolo_final(1)"
)

DATA_YAML = Path("/kaggle/working/varroa_data.yaml")

DATA_YAML.write_text(f"""path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

names:
  0: Varroa_mite
""")

print(DATA_YAML.read_text())

## 4. Train the final model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/kaggle/working/varroa_data.yaml",
    imgsz=640,
    epochs=50,
    batch=-1,
    seed=42,
    patience=10,
    device=0,

    # Experiment 5: emphasize bounding-box localization
    box=10.0,

    project="/kaggle/working/varroa_runs",
    name="yolov8n_box10",
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)

## 5. Validate the final model

In [ ]:
from ultralytics import YOLO

model = YOLO(
    "/kaggle/working/varroa_runs/yolov8s_640/weights/best.pt"
)

results = model.val(
    data="/kaggle/working/varroa_data.yaml",
    split="val",
    imgsz=640,
    device=0,
    plots=True,
    save=True,
    project="/kaggle/working/error_analysis",
    name="yolov8s_val"
)

print("\n" + "="*60)
print("YOLOv8s VALIDATION")
print("="*60)
print(f"Precision : {results.box.mp:.4f}")
print(f"Recall    : {results.box.mr:.4f}")
print(f"mAP50     : {results.box.map50:.4f}")
print(f"mAP50-95  : {results.box.map:.4f}")

## 6. Sample inference

Set `SAMPLE_IMAGE` to a real test image before running this cell.

In [ ]:
SAMPLE_IMAGE = "/path/to/test_image.jpg"

results = best_model.predict(
    source=SAMPLE_IMAGE,
    imgsz=640,
    conf=0.25,
    save=True
)

print("Inference completed.")


## 7. Deployment artifact

In [ ]:
BEST_MODEL = Path(
    "/kaggle/working/varroa_runs/yolov8s_640/weights/best.pt"
)

print("Best model:", BEST_MODEL)
print("Exists:", BEST_MODEL.exists())


### ML-service deployment

Copy the final:

```text
best.pt
```

to:

```text
ml_service/models/best.pt
```

The FastAPI service uses this model for:

```text
POST /predict/varroa
GET  /health/varroa
```

Do not commit the generated `varroa_runs/` directory or training images to the production repository.
